# Gap-filling metrics: hold-out runs for several gauges

Explores the runs of `find_best_model_holdout.py` (`performance_holdout/<gauge>/<run label>/`)
for several gauges: which model wins each selection metric, how often the metrics pick the
same winner, and how the winners look against the observed flow, inside and outside the
hidden block. This is a gauge-by-gauge look, not the full-pool comparison.

What it does not do: the referees (flow signatures, the day-by-day exceedance F1) are not
built yet, so these plots are for inspection, not a verdict. With few realizations per
model, small differences between winners are noise.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# The repo root holds the modules; the notebook runs with it as the working directory
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import find_best_model_holdout as fbh
import holdout_results as hr

warnings.filterwarnings("ignore", category=RuntimeWarning)  # overflow in poor parameter sets

GRID, SURFACE, INK = "#e1e0d9", "#fcfcfb", "#3b3b38"
SERIES_5 = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # categorical slots 1-5, fixed order


def style(fig, **layout):
    """The look of best_model_per_metric.ipynb: light surface, grid, horizontal legend on top."""
    fig.update_xaxes(showgrid=True, gridcolor=GRID, linecolor="#8a8984", zeroline=False)
    fig.update_yaxes(showgrid=True, gridcolor=GRID, linecolor="#8a8984", zeroline=False)
    fig.update_layout(template="simple_white", plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
                      font=dict(color=INK), margin=dict(t=110, l=70, r=30, b=60),
                      legend=dict(orientation="h", x=0, y=1.03, xanchor="left", yanchor="bottom",
                                  font=dict(size=11)), **layout)
    return fig


# ## Settings

def newest_finished_runs(root, max_gauges):
    """Per gauge folder under root, the newest run folder (by run_info.json time) whose status
    is "finished"; the max_gauges newest gauges, returned sorted by gauge id."""
    newest = {}  # gauge -> (time, run folder)
    for path in root.glob("*/*/run_info.json"):
        try:
            if json.loads(path.read_text()).get("status") != "finished":
                continue
        except (OSError, ValueError):
            continue
        gauge, when = path.parent.parent.name, path.stat().st_mtime
        if gauge not in newest or when > newest[gauge][0]:
            newest[gauge] = (when, path.parent)
    picked = sorted(newest.items(), key=lambda item: item[1][0], reverse=True)[:max_gauges]
    return [run_dir for _, (_, run_dir) in sorted(picked)]


MAX_GAUGES = 10
# Run folders: by default the newest finished run of each gauge. To pick them, give a list, e.g.
# RUN_DIRS = [fbh.PERFORMANCE_DIR / "03313700" / "smoke_03313700",
#             fbh.PERFORMANCE_DIR / "07351750" / "smoke_07351750"]
RUN_DIRS = newest_finished_runs(fbh.PERFORMANCE_DIR, MAX_GAUGES)

# Metrics drawn by default: five new ones, then NSE and KGE as the reference
CHOSEN_METRICS = ["KGE_NP", "KGE_TA_p5_p95", "KGE_TA_p5_p95_log", "PNSE_p5_p95", "JDKGE",
                  "NSE", "KGE"]
MODEL_FOR_SCATTER = None      # None = per gauge, the model that wins the first chosen metric with a winner
X_METRIC, Y_METRIC = None, None  # None = the first two chosen metrics
LOG_Y = True                  # log y-axis on the hydrographs
SKIP_SNOW = False             # not recorded in run_info; this is the default the runs used;
                              # change it only if the run was started with a different value

METRICS = list(fbh.SUMMARY_METRICS)  # the 15 metrics with a best value in summary.parquet
if not RUN_DIRS:
    raise FileNotFoundError(f"No finished run found under {fbh.PERFORMANCE_DIR}")
unknown = [m for m in CHOSEN_METRICS if m not in METRICS]
if unknown:
    raise ValueError(f"CHOSEN_METRICS has unknown names {unknown}; choose from {METRICS}")
X_METRIC = X_METRIC or CHOSEN_METRICS[0]
Y_METRIC = Y_METRIC or CHOSEN_METRICS[1]

In [ ]:
# ## Load the runs
# The gauge id is the name of the run folder's parent.

runs = {Path(run_dir).parent.name: hr.load_run(run_dir) for run_dir in RUN_DIRS}
GAUGES = list(runs)

rows = []
for gauge, run in runs.items():
    info = run["run_info"]
    row = {"gauge": gauge}
    for key in ["status", "start_year", "end_year", "holdout_intervals", "extreme_date",
                "extreme_value", "nonpositive_obs", "git_commit", "git_dirty"]:
        if key in info:
            row[key] = info[key]
    if "models_evaluated" in info:
        row["models evaluated"] = len(info["models_evaluated"])
    if "models_removed" in info:
        row["models removed"] = sorted(int(m) for m in info["models_removed"])
    rows.append(row)
display(pd.DataFrame(rows).set_index("gauge"))

for gauge, run in runs.items():
    status = run["run_info"].get("status")
    if status != "finished":
        print(f"WARNING gauge {gauge}: run status is {status!r}, not 'finished'; results may be incomplete")

In [ ]:
# ## Overview
# Share of the successful realizations (all models of a gauge pooled) where each metric is NaN;
# realizations per gauge, and how often the new metrics hit an unexpected exception.

shares = pd.DataFrame({m: [hr.nan_share(runs[g]["tables"], m) for g in GAUGES] for m in METRICS},
                      index=GAUGES)
fig = go.Figure(go.Heatmap(z=shares.to_numpy(), x=METRICS, y=GAUGES, zmin=0, zmax=1,
                           colorscale="Blues", text=shares.map(lambda v: f"{v:.2f}").to_numpy(),
                           texttemplate="%{text}", hoverongaps=False))
style(fig, title="Share of successful realizations with a NaN value", height=220 + 40 * len(GAUGES),
      yaxis=dict(type="category", autorange="reversed")).show()

counts = pd.DataFrame(
    {"n_ok": [int(runs[g]["summary"]["n_ok"].sum()) for g in GAUGES],
     "n_failed": [int(runs[g]["summary"]["n_failed"].sum()) for g in GAUGES],
     "metrics_error": [sum(int(t["metrics_error"].notna().sum())
                           for t in runs[g]["tables"].values() if "metrics_error" in t.columns)
                       for g in GAUGES]},
    index=pd.Index(GAUGES, name="gauge"))
display(counts)

In [ ]:
# ## Winners across gauges

best = {g: hr.best_model_per_metric(runs[g]["summary"], METRICS) for g in GAUGES}
cells = {g: {m: "none (all NaN)" if b["model"] is None else f"{b['model']} ({b['value']:.3f})"
             for m, b in best[g].items()} for g in GAUGES}
print("Winning model (its best value) per gauge and metric:")
display(pd.DataFrame(cells).T.rename_axis("gauge"))

for g in GAUGES:
    print(f"\ngauge {g}:")
    for model, won in hr.shared_winner_groups({m: b["model"] for m, b in best[g].items()}).items():
        print(f"  model {model} wins: {', '.join(won)}")
    no_winner = [m for m, b in best[g].items() if b["model"] is None]
    if no_winner:
        print(f"  no winner (all NaN): {', '.join(no_winner)}")

agree, both = hr.winner_agreement_counts(best, METRICS)
share = agree / both.where(both > 0)  # NaN where no gauge has both winners
fig = go.Figure(go.Heatmap(z=share.to_numpy(), x=METRICS, y=METRICS, zmin=0, zmax=1,
                           colorscale="Blues", text=(agree.astype(str) + "/" + both.astype(str)).to_numpy(),
                           texttemplate="%{text}", hoverongaps=False))
style(fig, title="Winner agreement between metrics across gauges (agree/both)", height=700,
      yaxis=dict(autorange="reversed")).show()
print("How to read it: each cell is agree/both, where both = gauges where the two metrics have a "
      "winner and agree = gauges where that winner is the same model (color = agree/both). "
      f"With {len(GAUGES)} gauge(s) these counts are thin.")

In [ ]:
# ## Rank correlation between the metrics
# Spearman correlation over the successful realizations of all models; choose a gauge or all
# gauges pooled in the dropdown. Blank cells: a metric that is NaN everywhere.

options = {"all gauges pooled": hr.metric_rank_correlation(
    {(g, m): t for g in GAUGES for m, t in runs[g]["tables"].items()}, METRICS)}
options.update({f"gauge {g}": hr.metric_rank_correlation(runs[g]["tables"], METRICS) for g in GAUGES})

fig = go.Figure()
for i, (label, corr) in enumerate(options.items()):
    fig.add_trace(go.Heatmap(z=corr.to_numpy(), x=METRICS, y=METRICS, zmin=-1, zmax=1,
                             colorscale="RdBu", name=label, visible=i == 0, hoverongaps=False,
                             text=corr.map(lambda v: "" if pd.isna(v) else f"{v:.2f}").to_numpy(),
                             texttemplate="%{text}"))
buttons = [dict(label=label, method="update",
                args=[{"visible": [j == i for j in range(len(options))]},
                      {"title.text": f"Spearman rank correlation between metrics: {label}"}])
           for i, label in enumerate(options)]
style(fig, title=f"Spearman rank correlation between metrics: {next(iter(options))}", height=720,
      yaxis=dict(autorange="reversed"),
      updatemenus=[dict(type="dropdown", buttons=buttons, active=0, x=1.0, xanchor="right",
                        y=1.12, yanchor="bottom")])
fig.show()

In [ ]:
# ## Re-simulation of the winners
# The winner of every one of the 15 metrics is simulated again from its saved parameters, with
# the data rebuilt by prepare_data from the arguments in run_info.json (once per gauge). Each
# distinct (model, realization) is simulated once; its NSE on the masked frame is compared
# with the stored value.
# No forcing hash exists yet: a changed forcing file is only caught by the NSE check below.
# prepare_data may download again if the stored daily file does not cover the full period
# (this happened for gauge 07030500, whose record starts in 2001).


def rebuild_data(run_info, skip_snow):
    """The data dict of the run: prepare_data with the arguments main() passed (from run_info)."""
    return fbh.prepare_data(run_info["gauge"], run_info["start_year"], run_info["end_year"],
                            run_info["warmup_years"], run_info["et_method"], skip_snow=skip_snow,
                            precip_source=run_info["precip_source"],
                            nasa_sampling=run_info["nasa_sampling"],
                            holdout_mode=run_info["holdout_mode"],
                            holdout_years=run_info["holdout_years"],
                            holdout_position=run_info["holdout_position"])


def row_params(model, row):
    """The parameters of a table row, as _run_one receives them (name -> float)."""
    return {name: float(row[name]) for name in fbh.marrmot.MODEL_REGISTRY[model]["param_names"]}


def resimulate(model, row, data):
    """(full, masked): the Q / Qobs frame as _run_one builds it, before and after hiding the block."""
    q = fbh.simulate_series(model, row_params(model, row), data)  # floored at Q_FLOOR
    full = pd.DataFrame({"Q": q, "Qobs": data["Qobs"]}, index=data["dates"])
    masked = fbh.hb.mask_hidden(full, data["hidden"]) if data["hidden"].any() else full
    return full, masked


def resimulate_winners(gauge, run, best, data):
    """({metric: entry}, {(model, realization): entry}); entry = {"model", "row", "frame", "full"}."""
    by_metric, distinct = {}, {}
    for metric in METRICS:
        model = best[metric]["model"]
        if model is None:
            print(f"gauge {gauge}, {metric}: no winner (all NaN), skipped")
            continue
        row = hr.best_row(run["tables"][model], metric)
        if row is None:
            print(f"gauge {gauge}, {metric}: model {model} has no valid row, skipped")
            continue
        key = (model, row.name)
        if key not in distinct:
            full, masked = resimulate(model, row, data)
            distinct[key] = {"model": model, "row": row, "frame": masked, "full": full}
        by_metric[metric] = distinct[key]
    return by_metric, distinct


def check_nse(entry, run_info, label):
    """(True, "OK ...") or (False, "WARNING ..."): NSE of the masked frame against the stored NSE."""
    mode = {"nonpositive": run_info["nonpositive_obs"], "offset_frac": run_info["obs_offset_frac"]}
    stored = float(entry["row"]["NSE"])
    try:
        again = float(fbh.pm.nse(entry["frame"], "Q", "Qobs", **mode))
    except ValueError as exc:
        again = float("nan")
        print(f"  ({label}: NSE could not be recomputed: {exc})")
    if np.isnan(stored) and np.isnan(again):
        return True, f"OK {label}: NSE is NaN in both"
    if abs(again - stored) <= 1e-9 * abs(stored):
        return True, f"OK {label}: NSE {again:.12g} equals the stored value"
    return False, (f"WARNING {label}: NSE {again!r} vs stored {stored!r}; the re-simulation or "
                   "the forcing differs from the run")


DATA, RESIMS = {}, {}  # gauge -> data dict; gauge -> metric -> {"model", "row", "frame", "full"}
for gauge in GAUGES:
    run = runs[gauge]
    try:
        data = rebuild_data(run["run_info"], SKIP_SNOW)
        by_metric, distinct = resimulate_winners(gauge, run, best[gauge], data)
    except Exception as exc:
        print(f"ERROR gauge {gauge}: {type(exc).__name__}: {exc}; gauge skipped")
        continue
    DATA[gauge], RESIMS[gauge] = data, by_metric
    results = []
    for (model, realization), entry in distinct.items():
        won = ", ".join(m for m, e in by_metric.items() if e is entry)
        results.append(check_nse(entry, run["run_info"],
                                 f"gauge {gauge}, model {model} realization {realization} ({won})"))
    n_ok = sum(ok for ok, _ in results)
    print(f"gauge {gauge}: {n_ok} OK, {len(results) - n_ok} WARNING")
    for ok, line in results:
        if not ok:
            print(line)

In [ ]:
# ## Hydrograph
# Per gauge: observed flow (training days, hidden days in a separate color), the hold-out block
# shaded and one line per metric winner. The dropdown chooses which winners are drawn.


def winner_colors(winners):
    """One color per distinct (model, realization), so a realization that wins several metrics
    keeps its color."""
    keys = list(dict.fromkeys((w["model"], w["row"].name) for w in winners.values()))
    return {k: SERIES_5[i % len(SERIES_5)] for i, k in enumerate(keys)}


def metric_menu(tags, winners, y=1.0):
    """Dropdown over the traces; tags[i] is "observed" or the metric whose winner trace i draws."""
    def visible(selected):
        return [tag == "observed" or tag in selected for tag in tags]

    buttons = [dict(label="Chosen metrics (all)", method="restyle",
                    args=[{"visible": visible(set(CHOSEN_METRICS))}])]
    for metric in METRICS:
        w = winners.get(metric)
        label = f"{metric} · model {w['model']}" if w else f"{metric} · no winner"
        buttons.append(dict(label=label, method="restyle",
                            args=[{"visible": visible({metric} if w else set())}]))
    buttons.append(dict(label="Observed only", method="restyle", args=[{"visible": visible(set())}]))
    return dict(type="dropdown", buttons=buttons, active=0, x=1.01, xanchor="left", y=y,
                yanchor="top")


def hydrograph(gauge, x_range=None, title=""):
    data, winners = DATA[gauge], RESIMS[gauge]
    obs = pd.Series(data["Qobs"], index=data["dates"])
    hidden = pd.Series(data["hidden"], index=data["dates"])
    colors = winner_colors(winners)
    fig, tags = go.Figure(), []
    fig.add_trace(go.Scatter(x=obs.index, y=obs.where(~hidden), name="Observed (training days)",
                             line=dict(color=INK, width=1.2)))
    fig.add_trace(go.Scatter(x=obs.index, y=obs.where(hidden), name="Observed (hidden days)",
                             line=dict(color="#c0392b", width=1.2)))
    tags += ["observed", "observed"]
    for metric, w in winners.items():
        fig.add_trace(go.Scatter(x=w["full"].index, y=w["full"]["Q"],
                                 name=f"{metric}: model {w['model']} r{w['row'].name}",
                                 line=dict(width=1, color=colors[(w["model"], w["row"].name)])))
        tags.append(metric)
    menu = metric_menu(tags, winners)
    for trace, shown in zip(fig.data, menu["buttons"][0]["args"][0]["visible"]):
        trace.visible = shown  # the initial view matches "Chosen metrics (all)"
    for start, end in runs[gauge]["run_info"].get("holdout_intervals") or []:
        fig.add_vrect(x0=start, x1=end, fillcolor="#8a8984", opacity=0.12, line_width=0,
                      layer="below")
    style(fig, title=title, height=620, hovermode="x unified", updatemenus=[menu])
    fig.update_layout(margin=dict(r=260))
    fig.update_yaxes(title_text="Flow (mm/day)", type="log" if LOG_Y else "linear")
    fig.update_xaxes(rangeslider_visible=True, range=x_range)
    return fig


for gauge in GAUGES:
    if gauge not in RESIMS:
        print(f"gauge {gauge}: not re-simulated, no hydrograph")
        continue
    hydrograph(gauge, title=f"Gauge {gauge}: observed and winners").show()
    intervals = runs[gauge]["run_info"].get("holdout_intervals") or []
    if intervals:
        hydrograph(gauge, [intervals[0][0], intervals[-1][1]],
                   f"Gauge {gauge}: the hold-out block").show()
    else:
        print(f"gauge {gauge}: no hold-out block, no zoomed figure")

In [ ]:
# ## Flow-duration curves
# Per gauge, training days (left) and hidden days (right), observed days only, the winners on
# the same days. Dashed lines: the 5/95 and 2/98 percent anchors; shaded: the default low and
# high blocks (the pdi_low and pdi_high percentile lists).

print("These are percentiles of the sorted flows, so they carry no timing.")


def exceedance_curve(values):
    """Exceedance probability (%) and flow, sorted from high to low (Weibull positions)."""
    q = np.sort(values)[::-1]
    return 100 * np.arange(1, len(q) + 1) / (len(q) + 1), q


def flow_duration(gauge):
    data, winners = DATA[gauge], RESIMS[gauge]
    obs = pd.Series(data["Qobs"], index=data["dates"])
    hidden = pd.Series(data["hidden"], index=data["dates"])
    colors = winner_colors(winners)
    panels = [("training days", obs.notna() & ~hidden), ("hidden days", obs.notna() & hidden)]
    fig = make_subplots(rows=1, cols=2, subplot_titles=[p for p, _ in panels], shared_yaxes=True)
    series = [("observed", "Observed", obs, INK)] + \
             [(m, f"{m}: model {w['model']} r{w['row'].name}", w["full"]["Q"],
               colors[(w["model"], w["row"].name)]) for m, w in winners.items()]
    tags = []
    for col, (panel, days) in enumerate(panels, 1):
        n_dropped = 0
        for tag, name, flow, color in series:
            values = flow[days].to_numpy(dtype=float)
            values = values[np.isfinite(values)]
            n_dropped += int((values <= 0).sum())
            x, y = exceedance_curve(values[values > 0])
            fig.add_trace(go.Scatter(x=x, y=y, name=name, legendgroup=name, showlegend=col == 1,
                                     line=dict(width=1.4, color=color)), row=1, col=col)
            tags.append(tag)
        print(f"gauge {gauge}, {panel}: {n_dropped} zero or negative value(s) not drawn on the log axis")
    menu = metric_menu(tags, winners)
    for trace, shown in zip(fig.data, menu["buttons"][0]["args"][0]["visible"]):
        trace.visible = shown
    # Anchors: percentile p sits at exceedance 100 - p (5/95 and 2/98 give the same lines)
    for p in (2, 5, 95, 98):
        fig.add_vline(x=p, line=dict(color="#8a8984", width=1, dash="dash"), row="all", col="all")
    low, high = fbh.gm.DEFAULT_LOW_BLOCK, fbh.gm.DEFAULT_HIGH_BLOCK
    for block, color in ((low, "#2a78d6"), (high, "#eb6834")):
        fig.add_vrect(x0=100 - max(block), x1=100 - min(block), fillcolor=color, opacity=0.08,
                      line_width=0, row="all", col="all")
    style(fig, title=f"Gauge {gauge}: flow-duration curves", height=560, updatemenus=[menu])
    fig.update_layout(margin=dict(r=260))
    fig.update_xaxes(title_text="Exceedance probability (%)", range=[0, 100])
    fig.update_yaxes(type="log")
    fig.update_yaxes(title_text="Flow (mm/day)", row=1, col=1)
    return fig


for gauge in GAUGES:
    if gauge not in RESIMS:
        print(f"gauge {gauge}: not re-simulated, no flow-duration curves")
        continue
    flow_duration(gauge).show()

In [ ]:
# ## Metric against metric
# Per gauge, all successful realizations of one model, with a star at the winning realization of
# every metric that has a winner (winners from other models at their own values). The two
# dropdowns change the x and y metric; the stars move with them.


def scatter_model_for(gauge):
    """MODEL_FOR_SCATTER if the gauge has it, else the winner of the first chosen metric with one."""
    if MODEL_FOR_SCATTER is not None:
        if MODEL_FOR_SCATTER in runs[gauge]["tables"]:
            return MODEL_FOR_SCATTER
        print(f"gauge {gauge}: model {MODEL_FOR_SCATTER} not in this run, using the default")
    return next((best[gauge][m]["model"] for m in CHOSEN_METRICS
                 if best[gauge][m]["model"] is not None), None)


def axis_menu(axis, points, initial, x_position):
    """Dropdown that sets only the `axis` data ("x" or "y") of every trace and the axis title."""
    buttons = [dict(label=f"{axis}: {metric}", method="update",
                    args=[{axis: [frame[metric].to_numpy(dtype=float) for frame in points]},
                          {f"{axis}axis.title.text": metric}])
               for metric in METRICS]
    return dict(type="dropdown", buttons=buttons, active=METRICS.index(initial), x=x_position,
                xanchor="left", y=1.12, yanchor="bottom")


for gauge in GAUGES:
    model = scatter_model_for(gauge)
    if model is None:
        print(f"gauge {gauge}: no chosen metric has a winner, no scatter")
        continue
    table = runs[gauge]["tables"][model]
    ok = table[table["error"].isna()]
    stars = {}  # metric -> one-row frame of its winning realization
    for metric in METRICS:
        winner = best[gauge][metric]["model"]
        if winner is None:
            continue
        row = hr.best_row(runs[gauge]["tables"][winner], metric)
        if row is not None:
            stars[metric] = (winner, row)
    keys = list(dict.fromkeys((w, r.name) for w, r in stars.values()))
    points = [ok] + [row.to_frame().T for _, row in stars.values()]
    fig = go.Figure(go.Scatter(x=ok[X_METRIC], y=ok[Y_METRIC], mode="markers",
                               name=f"model {model} realizations",
                               marker=dict(color="#b5b4ad", size=8), text=ok.index,
                               hovertemplate="realization %{text}<br>%{x:.3f}, %{y:.3f}<extra></extra>"))
    for metric, (winner, row) in stars.items():
        fig.add_trace(go.Scatter(x=[float(row[X_METRIC])], y=[float(row[Y_METRIC])], mode="markers",
                                 name=f"best {metric}: model {winner} r{row.name}",
                                 marker=dict(size=13, symbol="star",
                                             color=SERIES_5[keys.index((winner, row.name)) % len(SERIES_5)])))
    style(fig, title=f"Gauge {gauge}: metric against metric, model {model}", height=600,
          xaxis_title=X_METRIC, yaxis_title=Y_METRIC,
          updatemenus=[axis_menu("x", points, X_METRIC, 0.0), axis_menu("y", points, Y_METRIC, 0.3)])
    fig.update_layout(margin=dict(t=160))
    fig.show()

In [ ]:
# ## Metric values of the winners
# Rows: (gauge, metric whose winner it is); columns: the 15 metrics.

values = pd.DataFrame({(g, m): w["row"][METRICS].astype(float)
                       for g, by_metric in RESIMS.items() for m, w in by_metric.items()}).T
values.index.names = ["gauge", "metric"]
display(values)

## Referees (not built yet)

Flow signatures and the day-by-day exceedance F1 on the hidden days will be added here.
Until then, no conclusion about which metric is better should be drawn from the plots
above.